## This notebook joins word level videos to create sentence level videos

In [2]:
import json
import os
from pprint import pprint
import cv2
import pandas as pd
from tqdm import tqdm

In [3]:
json_path = "./app_asl_sentences.json"

with open(json_path, 'r') as f:
    data = json.load(f)

pprint(data)

{'sentences': [{'asl': 'hello you', 'english': 'Hello to you', 'id': 1},
               {'asl': 'hello friend', 'english': 'Hello friend', 'id': 2},
               {'asl': 'bye friend', 'english': 'Goodbye friend', 'id': 3},
               {'asl': 'thankyou you', 'english': 'Thank you', 'id': 4},
               {'asl': 'sorry friend', 'english': 'Sorry my friend', 'id': 5},
               {'asl': 'please help me', 'english': 'Please help me', 'id': 6},
               {'asl': 'what your name',
                'english': 'What is your name',
                'id': 7},
               {'asl': 'who you', 'english': 'Who are you', 'id': 8},
               {'asl': 'where you go',
                'english': 'Where are you going',
                'id': 9},
               {'asl': 'why you sad', 'english': 'Why are you sad', 'id': 10},
               {'asl': 'how you', 'english': 'How are you', 'id': 11},
               {'asl': 'when you arrive',
                'english': 'When will you arrive',


In [4]:
test_csv_path = r"D:\self\projects\masters\capstone\git\asl_trm\data\app\val_features.csv"

word_df = pd.read_csv(test_csv_path)
word_df["video_names"] = word_df["video_path"].apply(lambda x: x.split("/")[-1])
word_df.head(10)
# word_df["word"].unique()

,video_path,class_gloss,class_id,feature_path,video_names
0,ASL-Citizen/videos/657811957119824-CANNOT.mp4,cannot,91,features_cache/657811957119824-CANNOT.pt,657811957119824-CANNOT.mp4
1,ASL-Citizen/videos/6421905891904218-AFTER.mp4,after,98,features_cache/6421905891904218-AFTER.pt,6421905891904218-AFTER.mp4
2,ASL-Citizen/videos/217732808571379-YOUR.mp4,your,16,features_cache/217732808571379-YOUR.pt,217732808571379-YOUR.mp4
3,ASL-Citizen/videos/6131044630427722-REMEMBER.mp4,remember,63,features_cache/6131044630427722-REMEMBER.pt,6131044630427722-REMEMBER.mp4
4,ASL-Citizen/videos/047487125393855356-COMPUTER...,computer,89,features_cache/047487125393855356-COMPUTER.pt,047487125393855356-COMPUTER.mp4
5,ASL-Citizen/videos/20613001181771762-TOMORROW.mp4,tomorrow,41,features_cache/20613001181771762-TOMORROW.pt,20613001181771762-TOMORROW.mp4
6,ASL-Citizen/videos/13884914336494525-EAT.mp4,eat,50,features_cache/13884914336494525-EAT.pt,13884914336494525-EAT.mp4
7,ASL-Citizen/videos/6723858714545425-seedWHO.mp4,who,12,features_cache/6723858714545425-seedWHO.pt,6723858714545425-seedWHO.mp4
8,ASL-Citizen/videos/9811460119529283-HOW.mp4,how,33,features_cache/9811460119529283-HOW.pt,9811460119529283-HOW.mp4
9,ASL-Citizen/videos/9902995590504236-YOU.mp4,you,13,features_cache/9902995590504236-YOU.pt,9902995590504236-YOU.mp4


## use videos only in val set for stiching

In [5]:
video_dir = r"D:\self\projects\masters\capstone\data\testing\asl-100-citizen\new_videos"
# filtered video_paths based on val_df

val_video_names = word_df["video_names"].unique()

all_video_paths = os.listdir(video_dir)
print(f"""Total videos: {len(all_video_paths)}""") # 
filtered_video_paths = [video for video in all_video_paths if video in val_video_names]
print(f"""Filtered videos: {len(filtered_video_paths)}""")

Total videos: 3530
Filtered videos: 706


In [6]:
water = [video for video in filtered_video_paths[:] if "HELLO" in video]
water

['081380748015633-HELLO.mp4',
 '11225598264242453-HELLO.mp4',
 '1340650432799635-HELLO.mp4',
 '7472491282176557-HELLO.mp4',
 '7613696736100914-HELLO.mp4']

### stitch videos

In [ ]:
import os
import json
import cv2
import numpy as np
import random
import subprocess
from pathlib import Path

# -----------------------------
# Configuration
# -----------------------------
SENTENCE_JSON = "./app_asl_sentences.json"

OUTPUT_DIR = r"D:\self\projects\masters\capstone\data\testing\sentence-level\sentence-level-stitched"
VIDEO_DIR = r"D:\self\projects\masters\capstone\data\testing\asl-100-citizen\new_videos"

BASE_FPS = 30

MIN_PAUSE = 5
MAX_PAUSE = 20

MIN_SPEED = 0.7
MAX_SPEED = 1.3

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Collect all available videos
filtered_video_paths = os.listdir(VIDEO_DIR)
# -----------------------------
def read_video_frames(video_path):

    cap = cv2.VideoCapture(video_path)

    frames = []
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)

    cap.release()

    return frames, width, height


def create_blank_frames(width, height, count):

    blank = np.zeros((height, width, 3), dtype=np.uint8)

    return [blank.copy() for _ in range(count)]


def apply_speed_variation(frames):

    speed = random.uniform(MIN_SPEED, MAX_SPEED)

    if speed > 1:
        step = max(1, int(speed))
        frames = frames[::step]

    else:
        repeat = max(1, int(1 / speed))
        new_frames = []
        for f in frames:
            new_frames.extend([f] * repeat)
        frames = new_frames

    return frames


# -----------------------------
# CV2 Video Writer (Streamlit-compatible)
# -----------------------------
def write_video_cv2(frames, output_path, fps):
    """Write frames to MP4 using H.264 (avc1) for browser/Streamlit playback.
    Falls back to mp4v if avc1 is unavailable on the system."""
    height, width, _ = frames[0].shape

    for codec in ("avc1", "mp4v"):
        fourcc = cv2.VideoWriter_fourcc(*codec)
        writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
        if writer.isOpened():
            break
        writer.release()

    for frame in frames:
        writer.write(frame)

    writer.release()


# -----------------------------
# Load Dataset
# -----------------------------
with open(SENTENCE_JSON) as f:
    sentences = json.load(f)["sentences"]


# -----------------------------
# Generate Videos
# -----------------------------
for sentence in sentences:

    asl = sentence["asl"]
    print(f"\nProcessing: {asl}")

    words = asl.split()

    sentence_frames = []
    width = None
    height = None

    for i, word in enumerate(words):

        print(f"Processing word: {word.upper()}")

        # get videos related to word
        word_videos = [
            video for video in filtered_video_paths
            if word.upper() in video
        ]

        if not word_videos:
            print(f"Missing video for: {word}")
            break

        # randomly select video
        video_name = random.choice(word_videos)
        video_path = os.path.join(VIDEO_DIR, video_name)

        print(f"Selected video: {video_path}")

        frames, width, height = read_video_frames(video_path)

        # speed variation
        frames = apply_speed_variation(frames)

        sentence_frames.extend(frames)

        # random pause
        if i < len(words) - 1:

            pause = random.randint(MIN_PAUSE, MAX_PAUSE)

            blanks = create_blank_frames(width, height, pause)

            sentence_frames.extend(blanks)

    if not sentence_frames:
        continue

    output_name = asl.replace(" ", "_") + ".mp4"
    output_path = os.path.join(OUTPUT_DIR, output_name)

    write_video_cv2(sentence_frames, output_path, BASE_FPS)

    print(f"Saved: {output_path}")


Processing: hello you


Processing word: HELLO
Selected video: D:\self\projects\masters\capstone\data\testing\asl-100-citizen\new_videos\081380748015633-HELLO.mp4
Processing word: YOU
Selected video: D:\self\projects\masters\capstone\data\testing\asl-100-citizen\new_videos\06476990357042611-THANK YOU.mp4


FileNotFoundError: [WinError 2] The system cannot find the file specified